# 1 import necessary library

In [ ]:

# ============================================
# 1. IMPORT NECESSARY LIBRARIES
# ============================================

import tensorflow as tf
import keras
import numpy as np
import matplotlib.pyplot as plt

from keras import Sequential
from keras.layers import Conv2D, MaxPool2D, UpSampling2D


# ============================================
# 2. IMPORT COLOUR IMAGE DATASET
# ============================================

(X_train, _), (X_test, _) = keras.datasets.cifar10.load_data()

print("Original Train Shape:", X_train.shape)
print("Original Test Shape:", X_test.shape)


# ============================================
# 3. DATA PREPARATION
# ============================================

# Normalize pixel values from 0-255 to 0-1

X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

print("After Normalization:")
print(X_train.shape)
print(X_test.shape)


# ============================================
# 3.1 DISPLAY ORIGINAL COLOUR IMAGE
# ============================================

plt.figure(figsize=(5,5))
plt.imshow(X_train[2500])
plt.title("Original Colour Image")
plt.axis("off")
plt.show()


# ============================================
# 3.2 ADD NOISE / BLUR EFFECT
# ============================================

X_train_noise = X_train + 0.2 * np.random.normal(
    loc=0.0,
    scale=1.0,
    size=X_train.shape
)

X_test_noise = X_test + 0.2 * np.random.normal(
    loc=0.0,
    scale=1.0,
    size=X_test.shape
)


# ============================================
# 3.3 CLIP VALUES BETWEEN 0 AND 1
# ============================================

X_train_clipped = np.clip(
    X_train_noise,
    a_min=0.0,
    a_max=1.0
)

X_test_clipped = np.clip(
    X_test_noise,
    a_min=0.0,
    a_max=1.0
)


# ============================================
# 3.4 DISPLAY NOISY / BLUR-LIKE IMAGE
# ============================================

plt.figure(figsize=(5,5))
plt.imshow(X_train_clipped[2500])
plt.title("Noisy Colour Image")
plt.axis("off")
plt.show()


# ============================================
# 4. MODEL BUILDING
# ============================================

model = Sequential()

# --------------------------------------------
# ENCODER
# --------------------------------------------

model.add(
    Conv2D(
        filters=32,
        kernel_size=(3,3),
        strides=(1,1),
        padding="same",
        activation="relu",
        input_shape=(32,32,3)
    )
)

model.add(
    MaxPool2D(pool_size=(2,2))
)

model.add(
    Conv2D(
        filters=8,
        kernel_size=(3,3),
        strides=(1,1),
        padding="same",
        activation="relu"
    )
)

model.add(
    MaxPool2D(pool_size=(2,2))
)


# --------------------------------------------
# DECODER
# --------------------------------------------

model.add(
    Conv2D(
        filters=8,
        kernel_size=(3,3),
        strides=(1,1),
        padding="same",
        activation="relu"
    )
)

model.add(
    UpSampling2D(size=(2,2))
)

model.add(
    Conv2D(
        filters=32,
        kernel_size=(3,3),
        strides=(1,1),
        padding="same",
        activation="relu"
    )
)

model.add(
    UpSampling2D(size=(2,2))
)

# IMPORTANT:
# 3 filters because output is RGB colour image

model.add(
    Conv2D(
        filters=3,
        kernel_size=(3,3),
        strides=(1,1),
        padding="same",
        activation="sigmoid"
    )
)

model.summary()


# ============================================
# 5. MODEL COMPILATION
# ============================================

model.compile(
    optimizer="adam",
    loss="mean_squared_error",
    metrics=["accuracy"]
)


# ============================================
# 6. MODEL TRAINING
# ============================================

history = model.fit(
    x=X_train_clipped,
    y=X_train,
    batch_size=32,
    epochs=10,
    validation_data=(X_test_clipped, X_test)
)


# ============================================
# 7. MODEL EVALUATION
# ============================================

model.evaluate(
    X_test_clipped,
    X_test
)


# ============================================
# 8. MODEL PREDICTION
# ============================================

y_pred = model.predict(X_test_clipped)


# ============================================
# 9. SHOW RESULTS
# ============================================

index = 2500

plt.figure(figsize=(15,5))

# Original image
plt.subplot(1,3,1)
plt.imshow(X_test[index])
plt.title("Original Clear Image")
plt.axis("off")

# Noisy image
plt.subplot(1,3,2)
plt.imshow(X_test_clipped[index])
plt.title("Noisy / Blur Image")
plt.axis("off")

# Reconstructed image
plt.subplot(1,3,3)
plt.imshow(y_pred[index])
plt.title("AutoEncoder Output")
plt.axis("off")

plt.show()


# ============================================
# 10. SAVE MODEL
# ============================================

model.save("colour_denoising_autoencoder.h5")
'''
